# Masked-Diffusion BabyLM — Strict-Small (Evaluation)

A masked-diffusion denoiser is scored like a masked LM (per-token
pseudo-log-likelihood), so we evaluate it with the official pipeline's **`mlm`**
backend. We run:

1. Full zero-shot + GLUE fine-tuning on the **final** model (`main`).
2. Fast zero-shot on **every** `chck_NM` checkpoint.
3. `collate_preds` to build the submission file.

The model must already be public on the Hub (see `upload_pipeline.ipynb`).

In [ ]:
# Cell 1 — Parameters
MODEL_ID = "<your-user>/babylm-2026-strict-small-mdlm-seed42"
BACKEND  = "mlm"            # masked-diffusion is scored as a masked LM
TRACK    = "strict-small"
EVAL_REPO = "https://github.com/<your-user>/diffusion-babylm-eval.git"
DRIVE_EVAL_ROOT = "/content/drive/MyDrive/BabyLM/diffusion/results"  # where results are persisted

In [ ]:
# Cell 2 — Setup: mount Drive, clone repo, install eval deps, HF token
from google.colab import drive; drive.mount("/content/drive")
import os, subprocess
if not os.path.isdir("/content/diffusion-babylm-eval"):
    subprocess.run(["git", "clone", EVAL_REPO, "/content/diffusion-babylm-eval"], check=True)
%cd /content/diffusion-babylm-eval/strict
!pip install -q -r requirements.txt
!python -m scripts.download_evals   # downloads BLiMP/EWoK/COMPS/entity_tracking/GLUE
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets.", e)

In [ ]:
# Cell 3 — FULL zero-shot on the final checkpoint (main)
# Scripts live in strict/scripts/ but read data from strict/ (relative paths),
# so we run them from strict/ as `bash scripts/<name>.sh`.
!bash scripts/eval_zero_shot.sh $MODEL_ID $BACKEND

In [ ]:
# Cell 4 — FAST zero-shot on all chck_NM checkpoints (intermediate checkpoints)
# Loops chck_1M..chck_10M then chck_20M..chck_100M (strict-small stops at 100M).
!bash scripts/eval_zero_shot_fast_all_revisions.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 5 — GLUE fine-tuning on the final checkpoint (full eval requirement)
# Uses named args; writes per-task scores under strict/results/.
!bash scripts/eval_finetuning.sh --model_path $MODEL_ID --seed 42

In [ ]:
# Cell 6 — (Optional) diffusion-native scorer: ELBO and the layer-duplication
# variant, which the official pipeline cannot express. Writes predictions.json
# in the same layout so it is interchangeable with the mlm backend above.
%cd /content/diffusion-babylm-eval/diffusion
!python scripts/diffusion_eval_backend.py --model_path_or_name $MODEL_ID \
    --task blimp --data_path ../strict/evaluation_data/full_eval/blimp_filtered \
    --scoring elbo --layer_duplication_factor 2 --backend mlm --save_predictions

In [ ]:
# Cell 7 — Build the submission file (predictions scored server-side)
# collate_preds.sh already passes --fast (required for a full Challenge submission).
%cd /content/diffusion-babylm-eval/strict
!bash scripts/collate_preds.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 8 — Collect results into a CSV + persist everything to Drive
# strict/results/ is on the ephemeral Colab disk, so we (1) print the markdown
# table, (2) flatten scores into results_summary.csv, and (3) copy results/ +
# the submission zip to Drive so they survive disconnects and feed the paper.
import csv, glob, os, shutil, subprocess
%cd /content/diffusion-babylm-eval/strict

print(subprocess.run(["python", "scripts/print_results_table.py"],
                     capture_output=True, text=True).stdout)

rows = []  # (split, task, subtask_or_metric, score)
for rep in sorted(glob.glob("results/*/main/*/causal/*/*/best_temperature_report.txt")):
    p = rep.split("/")
    for i, line in enumerate(open(rep)):
        if line.strip().startswith("### AVERAGE"):
            try: rows.append(("zero_shot", p[-3], p[-2], float(open(rep).read().splitlines()[i+1].strip())))
            except Exception: pass
            break
for res in sorted(glob.glob("results/*/main/finetune/*/results.txt")):
    task = res.split("/")[-2]
    for line in open(res):
        k, _, v = line.partition(":")
        if k.strip() in ("accuracy", "f1", "mcc"):
            try: rows.append(("finetune", task, k.strip(), float(v.strip()) * 100))
            except Exception: pass

safe = MODEL_ID.replace("/", "__")
dst = f"{DRIVE_EVAL_ROOT}/{safe}"
os.makedirs(dst, exist_ok=True)
with open(f"{dst}/results_summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["split", "task", "metric", "score"]); w.writerows(rows)
if os.path.isdir("results"):
    shutil.copytree("results", f"{dst}/results", dirs_exist_ok=True)
for z in glob.glob("*.zip"):  # collate_preds writes the submission zip here
    shutil.copy2(z, dst)
print(f"\nSaved {len(rows)} scores -> {dst}/results_summary.csv (+ results/ and submission on Drive)")